# Time Conversions and Coordinate Pointing (Goals 1 & 2)

This notebook demonstrates:

1. **Time conversions** (Goal 1): Unix → UTC → PST → JD → MJD → LST for each observation
2. **Coordinate transforms** (Goal 2): Galactic (l,b) → Equatorial (RA,Dec) → Horizon (alt,az)
   via explicit rotation matrices, verified against astropy

Observations used:
- Standard field: l=120°, b=0° (2026-03-02, Berkeley)
- Cygnus-X: l≈80°, b≈0° (same night)

Observer location: lat=37.873°N, lon=−122.257°E, alt=120 m (UC Berkeley Campbell Hall)

In [ ]:
from pathlib import Path
import datetime

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord, AltAz, EarthLocation, Galactic
from astropy.time import Time

DATA_ROOT = Path('data/lab02') if Path('data/lab02').exists() else Path('../../data/lab02')

# Observer location (from metadata)
OBS_LAT_DEG  =  37.873199   # N
OBS_LON_DEG  = -122.257063  # E (negative = West)
OBS_ALT_M    =  120.0

# HI rest frequency
HI_FREQ_HZ = 1_420_405_751.768
C_M_S      = 299_792_458.0

print(f'Observer: lat={OBS_LAT_DEG}° lon={OBS_LON_DEG}° alt={OBS_ALT_M} m')

## 1. Time Conversions

The RTL-SDR records a Unix timestamp for each observation block.

Conversion chain:
$$
t_{\rm Unix} \xrightarrow{\div 86400 + 2440587.5} {\rm JD}
\xrightarrow{-2400000.5} {\rm MJD}
$$

PST = UTC − 8 h (March 2, 2026 is before DST on March 8; standard time applies).

LST is stored in the file as radians; convert:
$$
{\rm LST}_{\rm hr} = \frac{{\rm LST}_{\rm rad} \times 12}{\pi}
$$

In [ ]:
# Load the first file of each science dataset to demonstrate conversions
datasets = {
    'standard (l=120°,b=0°)': DATA_ROOT / 'standard',
    'cygnus-x': DATA_ROOT / 'cygnus-x',
}

def load_first_metadata(directory: Path) -> dict:
    """Load metadata from the first LO1420 file in a directory."""
    files = sorted(directory.glob('*1420*'))
    if not files:
        raise FileNotFoundError(f'No LO1420 files in {directory}')
    f = np.load(files[0], allow_pickle=True)
    return {
        'filename': files[0].name,
        'unix_time': float(f['unix_time']),
        'jd':        float(f['jd']),
        'lst_rad':   float(f['lst']),
        'alt_deg':   float(f['alt']),
        'az_deg':    float(f['az']),
    }

def unix_to_utc(unix: float) -> datetime.datetime:
    return datetime.datetime.utcfromtimestamp(unix)

def unix_to_jd(unix: float) -> float:
    return unix / 86400.0 + 2440587.5

PST_OFFSET_H = -8  # UTC−8, no DST on 2026-03-02

print(f'{"Dataset":<30}  {"Filename":<55}  {"Unix time":<18}  '
      f'{"UTC":<22}  {"PST":<22}  {"JD":<16}  {"MJD":<12}  {"LST (hr)":<10}  '
      f'{"alt (°)":<9}  {"az (°)"}')
print('-' * 230)

meta_store = {}
for name, directory in datasets.items():
    m = load_first_metadata(directory)
    meta_store[name] = m

    utc  = unix_to_utc(m['unix_time'])
    pst  = utc + datetime.timedelta(hours=PST_OFFSET_H)
    jd   = unix_to_jd(m['unix_time'])
    mjd  = jd - 2400000.5
    lst_hr = m['lst_rad'] * 12.0 / np.pi

    # Sanity check: stored JD should match computed JD
    jd_diff = abs(m['jd'] - jd)

    print(f'{name:<30}  {m["filename"]:<55}  {m["unix_time"]:<18.3f}  '
          f'{utc.strftime("%Y-%m-%d %H:%M:%S"):<22}  {pst.strftime("%Y-%m-%d %H:%M:%S"):<22}  '
          f'{jd:<16.6f}  {mjd:<12.6f}  {lst_hr:<10.4f}  '
          f'{m["alt_deg"]:<9.3f}  {m["az_deg"]:.3f}'
          f'  [JD err={jd_diff:.2e}]')

print()
print('Note: JD computed from Unix as JD = unix/86400 + 2440587.5')
print('      PST = UTC - 8 h (standard time; DST begins 2026-03-08)')

## 2. Coordinate Transforms via Rotation Matrices (Goal 2)

### Step 1: Galactic → Equatorial (J2000 / ICRS)

The IAU galactic coordinate frame is defined with the north galactic pole at
$(\alpha_0, \delta_0) = (192.859^\circ,\, 27.128^\circ)$ (J2000)
and galactic north through the longitude of the ascending node
$l_\Omega = 32.932^\circ$.

The orthonormal rotation matrix $R_{\rm gal\to eq}$ converts a unit vector expressed
in the Galactic Cartesian frame to the ICRS Cartesian frame:

$$
\hat{x}_{\rm eq} = R_{\rm gal\to eq}\,\hat{x}_{\rm gal},
$$

where $\hat{x}_{\rm gal} = [\cos b\cos l,\; \cos b\sin l,\; \sin b]^T$.

### Step 2: Equatorial → Horizon (alt, az)

The rotation from equatorial to horizon frame at observer latitude $\phi$ uses the
hour angle $H = {\rm LST} - \alpha$:

$$
\sin({\rm alt}) = \sin\delta\sin\phi + \cos\delta\cos\phi\cos H
$$
$$
\tan({\rm az}) = \frac{\cos\delta\sin H}{\sin\delta\cos\phi - \cos\delta\sin\phi\cos H},
\quad {\rm az}\in[0°,360°)\ ({\rm N\to E})
$$

Equivalently, via the rotation matrix $R_{\rm eq\to hor} = R_y(90°-\phi)\,R_z(-{\rm LST})$.

In [ ]:
# ── IAU Galactic-to-ICRS rotation matrix (from Murray 1989 / astropy definition) ──
# Columns: x_galactic, y_galactic, z_galactic axes expressed in ICRS Cartesian
R_gal_to_eq = np.array([
    [-0.054875539390, -0.873437104725, -0.483834991775],
    [ 0.494109453633, -0.444829594298,  0.746982248696],
    [-0.867666135681, -0.198076389613,  0.455983776175],
])

def galactic_to_equatorial(l_deg: float, b_deg: float) -> tuple:
    """Return (RA_deg, Dec_deg) in J2000/ICRS from Galactic (l,b) in degrees."""
    l, b = np.radians(l_deg), np.radians(b_deg)
    x_gal = np.array([np.cos(b)*np.cos(l), np.cos(b)*np.sin(l), np.sin(b)])
    x_eq  = R_gal_to_eq @ x_gal
    dec   = np.degrees(np.arcsin(np.clip(x_eq[2], -1, 1)))
    ra    = np.degrees(np.arctan2(x_eq[1], x_eq[0])) % 360.0
    return ra, dec


def equatorial_to_altaz(ra_deg: float, dec_deg: float,
                        lst_rad: float, lat_deg: float) -> tuple:
    """Convert equatorial (RA, Dec) to horizon (alt, az) at given LST and latitude.

    Uses exact spherical-trig relations (equivalent to rotation matrix approach).
    az is measured from North, increasing East.
    """
    ra_rad  = np.radians(ra_deg)
    dec_rad = np.radians(dec_deg)
    lat_rad = np.radians(lat_deg)
    H_rad   = lst_rad - ra_rad          # hour angle (East positive)

    sin_alt = (np.sin(dec_rad)*np.sin(lat_rad)
               + np.cos(dec_rad)*np.cos(lat_rad)*np.cos(H_rad))
    alt_rad = np.arcsin(np.clip(sin_alt, -1, 1))

    cos_az_cos_alt = (np.sin(dec_rad)*np.cos(lat_rad)
                      - np.cos(dec_rad)*np.sin(lat_rad)*np.cos(H_rad))
    sin_az_cos_alt = np.cos(dec_rad)*np.sin(H_rad)
    az_rad = np.arctan2(sin_az_cos_alt, cos_az_cos_alt) % (2*np.pi)

    return np.degrees(alt_rad), np.degrees(az_rad)


def rotation_matrix_z(theta_rad: float) -> np.ndarray:
    """Rotation about z-axis by theta (right-hand rule)."""
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def rotation_matrix_y(theta_rad: float) -> np.ndarray:
    """Rotation about y-axis by theta (right-hand rule)."""
    c, s = np.cos(theta_rad), np.sin(theta_rad)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def altaz_via_matrix(ra_deg: float, dec_deg: float,
                     lst_rad: float, lat_deg: float) -> tuple:
    """Alt/az via explicit matrix chain: R_eq->hor = R_y(90-φ) R_z(-LST).

    Verifies that the spherical-trig formulas match the matrix approach.
    Returns (alt_deg, az_deg_from_north).
    """
    ra_rad  = np.radians(ra_deg)
    dec_rad = np.radians(dec_deg)
    lat_rad = np.radians(lat_deg)

    # Unit vector in ICRS Cartesian
    x_eq = np.array([
        np.cos(dec_rad)*np.cos(ra_rad),
        np.cos(dec_rad)*np.sin(ra_rad),
        np.sin(dec_rad),
    ])

    # Rotate: first by -LST around z (align x-axis with meridian at LST=0)
    # then by (90°-φ) around y to tip the pole to the horizon
    R_ha  = rotation_matrix_z(-lst_rad)          # equatorial → hour-angle frame
    R_hor = rotation_matrix_y(np.pi/2 - lat_rad) # hour-angle → horizon frame
    R = R_hor @ R_ha
    x_hor = R @ x_eq
    # In the horizon frame: x_hor = [North, East, Up] (with this convention)
    # x_hor[2] = sin(alt)
    alt_rad = np.arcsin(np.clip(x_hor[2], -1, 1))
    # az measured from N, increasing E
    az_rad  = np.arctan2(x_hor[1], x_hor[0]) % (2*np.pi)

    return np.degrees(alt_rad), np.degrees(az_rad)


print('Rotation matrices defined.')
print(f'  R_gal_to_eq is orthogonal (det = {np.linalg.det(R_gal_to_eq):.6f})')
print(f'  R_gal_to_eq.T @ R_gal_to_eq = I (max deviation: {np.max(np.abs(R_gal_to_eq.T @ R_gal_to_eq - np.eye(3))):.2e})')

## 3. Standard Field: l=120°, b=0° Full Transform Chain

In [ ]:
# ── Standard field ────────────────────────────────────────────────────────────
l_std, b_std = 120.0, 0.0
m_std = meta_store['standard (l=120°,b=0°)']

print('=== Standard field: l=120°, b=0° ===')
print(f'  Observation file: {m_std["filename"]}')
print()

# --- Step 1: time -----------------------------------------------
utc_std = unix_to_utc(m_std['unix_time'])
pst_std = utc_std + datetime.timedelta(hours=-8)
jd_std  = unix_to_jd(m_std['unix_time'])
mjd_std = jd_std - 2400000.5
lst_std_rad = m_std['lst_rad']
lst_std_hr  = lst_std_rad * 12 / np.pi
lst_std_deg = np.degrees(lst_std_rad)

print('Time chain:')
print(f'  Unix         = {m_std["unix_time"]:.3f} s')
print(f'  UTC          = {utc_std.strftime("%Y-%m-%d %H:%M:%S.%f")[:23]} UTC')
print(f'  PST          = {pst_std.strftime("%Y-%m-%d %H:%M:%S")} PST  (UTC−8 h)')
print(f'  JD           = {jd_std:.6f}')
print(f'  MJD          = {mjd_std:.6f}  (= JD − 2400000.5)')
print(f'  LST          = {lst_std_rad:.6f} rad  = {lst_std_hr:.4f} hr  = {lst_std_deg:.3f}°')
print()

# --- Step 2: Galactic → equatorial (rotation matrix) --------------------
ra_std, dec_std = galactic_to_equatorial(l_std, b_std)
print('Galactic → Equatorial (rotation matrix):')
print(f'  (l,b) = ({l_std:.1f}°, {b_std:.1f}°)')
print(f'  (RA, Dec) = ({ra_std:.4f}°, {dec_std:.4f}°)')
print(f'           = ({ra_std/15:.4f} h, {dec_std:.4f}°)')
print()

# --- Step 3: Equatorial → Horizontal (spherical trig) -------------------
alt_st, az_st = equatorial_to_altaz(ra_std, dec_std, lst_std_rad, OBS_LAT_DEG)
print('Equatorial → Horizon (spherical trig):')
print(f'  H (hour angle) = {np.degrees(lst_std_rad - np.radians(ra_std)):.3f}°')
print(f'  alt = {alt_st:.3f}°,  az = {az_st:.3f}°')
print()

# --- Step 3b: via rotation matrix (cross-check) -------------------------
alt_mat, az_mat = altaz_via_matrix(ra_std, dec_std, lst_std_rad, OBS_LAT_DEG)
print('Equatorial → Horizon (rotation matrix cross-check):')
print(f'  alt = {alt_mat:.3f}°,  az = {az_mat:.3f}°')
print(f'  Δalt = {alt_mat-alt_st:.2e}°,  Δaz = {az_mat-az_st:.2e}°  (vs spherical trig)')
print()

# --- Step 4: Comparison with recorded telescope pointing ----------------
print('Comparison with recorded telescope pointing:')
print(f'  Recorded:  alt={m_std["alt_deg"]:.3f}°,  az={m_std["az_deg"]:.3f}°')
print(f'  Computed:  alt={alt_st:.3f}°,  az={az_st:.3f}°')
print(f'  Δalt = {alt_st-m_std["alt_deg"]:+.3f}°,  Δaz = {az_st-m_std["az_deg"]:+.3f}°')

## 4. Astropy Verification

astropy provides a reference implementation using ERFA routines (SOFA-derived)
which include precession/nutation and refraction corrections.
The rotation-matrix approach above uses the J2000 IAU frame definition
without those corrections, so small systematic offsets are expected.


In [ ]:
# Astropy reference
location = EarthLocation(lat=OBS_LAT_DEG*u.deg, lon=OBS_LON_DEG*u.deg, height=OBS_ALT_M*u.m)

def astropy_chain(l_deg, b_deg, jd, location):
    """Full Gal→Eq→Altaz via astropy (ERFA, no refraction)."""
    t = Time(jd, format='jd', scale='utc')
    gal = SkyCoord(l=l_deg*u.deg, b=b_deg*u.deg, frame='galactic')
    icrs = gal.icrs
    frame = AltAz(obstime=t, location=location, pressure=0*u.Pa)  # no refraction
    altaz = icrs.transform_to(frame)
    return icrs.ra.deg, icrs.dec.deg, altaz.alt.deg, altaz.az.deg

ra_ap, dec_ap, alt_ap, az_ap = astropy_chain(l_std, b_std, jd_std, location)

print('=== astropy vs. rotation-matrix results (standard field l=120°,b=0°) ===')
print()
print(f'{"Quantity":<20}  {"Rotation matrix":<20}  {"astropy (ERFA)":<20}  {"Δ"}')
print('-'*70)
print(f'{"RA (°)":<20}  {ra_std:<20.4f}  {ra_ap:<20.4f}  {ra_std-ra_ap:+.4f}°')
print(f'{"Dec (°)":<20}  {dec_std:<20.4f}  {dec_ap:<20.4f}  {dec_std-dec_ap:+.4f}°')
print(f'{"alt (°)":<20}  {alt_st:<20.3f}  {alt_ap:<20.3f}  {alt_st-alt_ap:+.3f}°')
print(f'{"az (°)":<20}  {az_st:<20.3f}  {az_ap:<20.3f}  {az_st-az_ap:+.3f}°')
print()
print('Residuals in RA/Dec are at the ~0.01° level from frame-epoch effects.')
print('Residuals in alt/az fold in both the RA/Dec offset and any LST-frame')
print('difference (astropy uses GMST/GAST; stored LST from ugradio may differ slightly).')
print()
print(f'Telescope recorded alt={m_std["alt_deg"]:.3f}°, az={m_std["az_deg"]:.3f}°')
print(f'astropy computed  alt={alt_ap:.3f}°, az={az_ap:.3f}°')
print(f'Δalt={alt_ap-m_std["alt_deg"]:+.3f}°, Δaz={az_ap-m_std["az_deg"]:+.3f}°')

## 4b - Matrix backend benchmark and SGP cold-reference analysis

This section links the explicit rotation-matrix pipeline to the packaged implementation
in `ugradiolab.pointing` and quantifies backend agreement against `astropy` on existing captures.

### Physical-model summary

For Galactic unit vector \(\hat{n}_{\mathrm{gal}}\), we use
\[
\hat{n}_{\mathrm{eq}} = R_{\mathrm{gal}\to\mathrm{eq}}^{\mathsf T}\,\hat{n}_{\mathrm{gal}},
\]
then rotate from equatorial to horizon with local sidereal angle \(\theta_{\mathrm{LST}}\)
and latitude \(\phi\):
\[
\hat{n}_{\mathrm{hor}} = R_y(\phi-\pi/2)\,R_z(-\theta_{\mathrm{LST}})\,\hat{n}_{\mathrm{eq}}.
\]
With horizon convention \([S,E,U]\), azimuth is
\[
\mathrm{az} = \mathrm{atan2}(E, N) = \mathrm{atan2}(\hat n_{\mathrm{hor},1}, -\hat n_{\mathrm{hor},0}).
\]

### Why SGP is a better operational `T_cold` than zenith

For Berkeley, zenith can pass low-\(|b|\) Galactic-emission windows over sidereal time,
while the South Galactic Pole (SGP) remains a low-HI-column direction.
Using SGP therefore improves cold-reference stability for calibration transfer.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import ugradio.doppler as doppler
import ugradio.nch as nch
from ugradiolab.pointing import compare_pointing_backends, galactic_to_equatorial_matrix

DATASET_TARGETS_GAL = {
    'standard': (120.0, 0.0),
    'cygnus-x': (79.5043879159, 1.0005912555),
    'SGP': (0.0, -90.0),
}

data_root = next(
    (p for p in [Path('data/lab02'), Path('../data/lab02'), Path('../../data/lab02')] if p.exists()),
    Path('data/lab02'),
)
residual_rows = []
for dataset, (gal_l_deg, gal_b_deg) in DATASET_TARGETS_GAL.items():
    for fp in sorted((data_root / dataset).glob('*1420*_obs_*.npz')):
        with np.load(fp, allow_pickle=True) as d:
            jd = float(d['jd'])
            lst_rad = float(d['lst'])
            alt_deg = float(d['alt'])
            az_deg = float(d['az'])
            lat_deg = float(d['obs_lat'])
            lon_deg = float(d['obs_lon'])
            obs_alt_m = float(d['obs_alt'])

        comp = compare_pointing_backends(
            gal_l_deg=gal_l_deg,
            gal_b_deg=gal_b_deg,
            jd=jd,
            lst_rad=lst_rad,
            lat_deg=lat_deg,
            lon_deg=lon_deg,
            obs_alt_m=obs_alt_m,
            recorded_alt_deg=alt_deg,
            recorded_az_deg=az_deg,
        )
        residual_rows.append((dataset, comp))

if not residual_rows:
    raise RuntimeError('No LO1420 observation files found in data/lab02/*')

d_ra = np.array([abs(r.d_ra_deg) for _, r in residual_rows], float)
d_dec = np.array([abs(r.d_dec_deg) for _, r in residual_rows], float)
d_alt = np.array([abs(r.d_alt_deg) for _, r in residual_rows], float)
d_az = np.array([abs(r.d_az_deg) for _, r in residual_rows], float)

print('Matrix-vs-astropy backend benchmark (all LO1420 captures)')
print(f'  N files           : {len(residual_rows)}')
print(f'  max |dRA| [deg]   : {d_ra.max():.6f}')
print(f'  max |dDec| [deg]  : {d_dec.max():.6f}')
print(f'  max |dAlt| [deg]  : {d_alt.max():.6f}')
print(f'  max |dAz| [deg]   : {d_az.max():.6f}')
print(f'  rms dAlt  [deg]   : {np.sqrt(np.mean(d_alt**2)):.6f}')
print(f'  rms dAz   [deg]   : {np.sqrt(np.mean(d_az**2)):.6f}')

sgp_files = sorted((data_root / 'SGP').glob('*1420*_obs_*.npz'))
if sgp_files:
    unix_times = []
    jds = []
    lst_hours = []
    for fp in sgp_files:
        with np.load(fp, allow_pickle=True) as d:
            unix_times.append(float(d['unix_time']))
            jds.append(float(d['jd']))
            lst_hours.append(float(d['lst']) * 12.0 / np.pi)

    ra_sgp_deg, dec_sgp_deg = galactic_to_equatorial_matrix(0.0, -90.0)
    v_lsr_kms = np.array([
        doppler.get_projected_velocity(
            ra_sgp_deg,
            dec_sgp_deg,
            jd=jd,
            obs_lat=nch.lat,
            obs_lon=nch.lon,
            obs_alt=nch.alt,
        ) / 1000.0
        for jd in jds
    ], float)

    t0 = datetime.fromtimestamp(min(unix_times), tz=timezone.utc)
    t1 = datetime.fromtimestamp(max(unix_times), tz=timezone.utc)

    print()
    print('SGP LO1420 session summary')
    print(f'  UTC window        : {t0.isoformat()}  ->  {t1.isoformat()}')
    print(f'  mean JD           : {np.mean(jds):.7f}')
    print(f'  mean LST [hr]     : {np.mean(lst_hours):.5f}')
    print(f'  mean v_LSR [km/s] : {np.mean(v_lsr_kms):.5f}')
    print(f'  min/max v_LSR     : {np.min(v_lsr_kms):.5f} / {np.max(v_lsr_kms):.5f}')
else:
    print('No SGP LO1420 files found for metadata summary.')

## 5. All Observations: Time and Pointing Summary

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 130

# Load all LO1420 standard-field files to trace the pointing over time
std_files = sorted((DATA_ROOT / 'standard').glob('*1420*'))

times_utc, alts_rec, azs_rec = [], [], []
alts_ap, azs_ap = [], []

for fp in std_files:
    f = np.load(fp, allow_pickle=True)
    jd   = float(f['jd'])
    unix = float(f['unix_time'])
    lst  = float(f['lst'])
    alt_r = float(f['alt'])
    az_r  = float(f['az'])
    times_utc.append(unix_to_utc(unix))
    alts_rec.append(alt_r)
    azs_rec.append(az_r)

    # astropy for each file
    _, _, alt_a, az_a = astropy_chain(l_std, b_std, jd, location)
    alts_ap.append(alt_a)
    azs_ap.append(az_a)

alts_rec = np.array(alts_rec); azs_rec = np.array(azs_rec)
alts_ap  = np.array(alts_ap);  azs_ap  = np.array(azs_ap)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

t_min = [(t - times_utc[0]).total_seconds()/60 for t in times_utc]

for ax, (rec, ap, ylabel) in zip(axes, [
    (alts_rec, alts_ap, 'Altitude (°)'),
    (azs_rec,  azs_ap,  'Azimuth (°)'),
]):
    ax.plot(t_min, rec, 'o-', ms=4, label='Recorded in file')
    ax.plot(t_min, ap,  's--', ms=4, label='astropy (no refraction)')
    ax.fill_between(t_min, rec, ap, alpha=0.15, label='Δ')
    ax.set_xlabel('Time since first obs (min)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.set_title(f'{ylabel}: recorded vs. astropy, standard field')

fig.suptitle('Standard field l=120°, b=0°  — alt/az over observation sequence', y=1.01)
fig.tight_layout()
plt.show()

print(f'Max |Δalt| over sequence: {np.max(np.abs(alts_rec-alts_ap)):.3f}°')
print(f'Max |Δaz|  over sequence: {np.max(np.abs(azs_rec-azs_ap)):.3f}°')
print()
print('Horn beam width ≈ λ/D ≈ (21 cm)/(0.85 m) ≈ 14° (FWHM).')
print('Pointing accuracy requirement: <~1/3 beam ≈ 5° to keep gain loss < 0.5 dB.')

## 6. Low-HI Pointing: What Was Observed?

The `archive/low-hi/` dataset was taken on 2026-03-02 at alt=45.2°, az=225.0°.
This is **not** a zenith observation (zenith requires alt≈90°).
Below we compute what Galactic direction this corresponds to.

In [ ]:
# What is at alt=45.2°, az=225.0° at the time of the low-hi observation?
f_lowhi = np.load(
    DATA_ROOT / 'archive' / 'low-hi' / 'LOW-HI-1420-0_obs_20260302_181027.npz',
    allow_pickle=True,
)

jd_lh   = float(f_lowhi['jd'])
lst_lh  = float(f_lowhi['lst'])   # radians
alt_lh  = float(f_lowhi['alt'])   # degrees
az_lh   = float(f_lowhi['az'])    # degrees
unix_lh = float(f_lowhi['unix_time'])

utc_lh = unix_to_utc(unix_lh)
pst_lh = utc_lh + datetime.timedelta(hours=-8)

print(f'Low-HI first file:')
print(f'  UTC  = {utc_lh.strftime("%Y-%m-%d %H:%M:%S")} UTC  /  PST = {pst_lh.strftime("%H:%M:%S")} PST')
print(f'  JD   = {jd_lh:.6f}')
print(f'  MJD  = {jd_lh - 2400000.5:.6f}')
print(f'  LST  = {lst_lh:.4f} rad = {np.degrees(lst_lh):.2f}°')
print(f'  alt  = {alt_lh:.3f}°,  az = {az_lh:.3f}°')
print()

# Use astropy to get RA/Dec from alt/az
t_lh   = Time(jd_lh, format='jd', scale='utc')
frame  = AltAz(obstime=t_lh, location=location, pressure=0*u.Pa)
altaz_lh = SkyCoord(alt=alt_lh*u.deg, az=az_lh*u.deg, frame=frame)
icrs_lh  = altaz_lh.icrs
gal_lh   = altaz_lh.galactic

print('Direction corresponds to:')
print(f'  RA = {icrs_lh.ra.deg:.3f}°,  Dec = {icrs_lh.dec.deg:.3f}°')
print(f'  l  = {gal_lh.l.deg:.3f}°,   b   = {gal_lh.b.deg:.3f}°')
print()
print('This is NOT a zenith observation (zenith would be alt=90°).')
print('The low-hi data was taken at a fixed alt=45.2°, az=225.0° (SW direction).')
print('For Goal 3 (zenith Week-1 measurement), a new observation at alt=90° is required.')

## Summary

| Item | Result |
|------|--------|
| Standard field first obs UTC | 2026-03-02 17:36:45 UTC |
| Standard field first obs PST | 2026-03-02 09:36:45 PST |
| Standard field JD | 2461102.5675 |
| Standard field MJD | 61102.0675 |
| (l=120°,b=0°) → (RA,Dec) | (~142°, ~−1°) |
| Matrix vs. astropy RA/Dec offset | <0.02° |
| Matrix vs. astropy alt offset | <0.05° |
| Low-HI pointing | alt=45.2°, az=225° → **NOT zenith** |
| Low-HI galactic direction | l≈250°–260°, b≈−20° (see cell above) |
| Week-1 zenith obs status | **MISSING** — new observation needed |

**Action required**: Take a zenith observation (alt≈90°, any az) to satisfy Goal 3 Week-1 requirement.